# ห้องปฏิบัติการ: ต้นไม้การตัดสินใจ (Dicision Tree)

ในสมุดบันทึกนี้ คุณจะเห็นภาพว่าต้นไม้การตัดสินใจถูกแยกออกโดยใช้การเพิ่มข้อมูล

เราจะกลับมาดูชุดข้อมูลที่ใช้ในวิดีโอการบรรยาย ชุดข้อมูลคือ:

ตามที่คุณเห็นในบทเรียน ในต้นไม้การตัดสินใจ เราตัดสินใจว่าโหนดจะถูกแยกหรือไม่โดยดูที่ข้อมูลที่ได้รับ** (Information Gain) ที่การแยกนั้นจะให้เรา (ภาพของวิดีโอ IG)

โดยที่

$$\text{Information Gain} = H(p_1^\text{node})- \left(w^{\text{left}}H\left(p_1^\text{left}\right) + w^{\text{right}}H\left(p_1^\text{right}\right)\right),$$

และ  $H$ คือเอนโทรปี กำหนดเป็น

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$

โปรดจำไว้ว่า log ที่นี่ถูกกำหนดให้เป็นฐาน 2 เรียกใช้บล็อกโค้ดด้านล่างเพื่อดูด้วยตัวคุณเองว่าเอนโทรปี $H(p)$  มีพฤติกรรมอย่างไรในขณะที่ $p$ มีการเปลี่ยนแปลง

โปรดทราบว่า H จะบรรลุค่าสูงสุดเมื่อ $p = 0.5$.ซึ่งหมายความว่าความน่าจะเป็นของเหตุการณ์คือ  $0.5$. และค่าต่ำสุดของมันจะบรรลุได้ใน  $p = 0$  และ  $p = 1$ กล่าวคือ ความน่าจะเป็นของเหตุการณ์ที่เกิดขึ้นนั้นสามารถคาดเดาได้อย่างสมบูรณ์ ดังนั้น เอนโทรปีจึงแสดงระดับความสามารถในการคาดเดาของเหตุการณ์

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
try:
  %matplotlib widget
  print("widget is already installed")
except:
  print("widget is not been installed, install now..")
  !pip install ipympl

In [ ]:
!git clone https://github.com/Smith-WeStrideTH/Advance_Learning_Algorithm_Course.git
%cd Advance_Learning_Algorithm_Course/work 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils_c7 import *

In [ ]:
%matplotlib widget
_ = plot_entropy()


|                                                     |   Ear Shape | Face Shape | Whiskers |   Cat  |
|:---------------------------------------------------:|:---------:|:-----------:|:---------:|:------:|
| <img src="images/0.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Present  |    1   |
| <img src="images/1.png" alt="drawing" width="50"/> |   Floppy   |  Not Round  |  Present  |    1   |
| <img src="images/2.png" alt="drawing" width="50"/> |   Floppy   |  Round      |  Absent   |    0   |
| <img src="images/3.png" alt="drawing" width="50"/> |   Pointy   |  Not Round  |  Present  |    0   |
| <img src="images/4.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Present  |    1   |
| <img src="images/5.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Absent   |    1   |
| <img src="images/6.png" alt="drawing" width="50"/> |   Floppy   |  Not Round  |  Absent   |    0   |
| <img src="images/7.png" alt="drawing" width="50"/> |   Pointy   |  Round      |  Absent   |    1   |
| <img src="images/8.png" alt="drawing" width="50"/> |    Floppy  |   Round     |  Absent   |    0   |
| <img src="images/9.png" alt="drawing" width="50"/> |   Floppy   |  Round      |  Absent   |    0   |


เราจะใช้การเข้ารหัสแบบ **one-hot** เพื่อเข้ารหัสคุณลักษณะเชิงหมวดหมู่ จะเป็นดังนี้:

- รูปร่างหู: แหลม = 1, หย่อน = 0
- รูปหน้า: กลม = 1, ไม่กลม = 0
- หนวด: มี = 1, ไม่มี = 0

ดังนั้น เราจะมีสองชุด:
- `X_train`: สำหรับแต่ละตัวอย่าง จะมี 3 คุณลักษณะ:
            - รูปร่างหู (1 ถ้าแหลม 0 ถ้าไม่เป็นเช่นนั้น)
            - รูปหน้า (1 ถ้ากลม 0 ถ้าไม่เป็นเช่นนั้น)
            - หนวด (1 ถ้ามี 0 ถ้าไม่มี)


            
- `y_train`: สัตว์นั้นเป็นแมวหรือไม่
            - 1 ถ้าสัตว์นั้นเป็นแมว
            - 0 ถ้าไม่เป็นเช่นนั้น

In [ ]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [ ]:
#For instance, the first example
X_train[0]

หมายความว่าตัวอย่างแรกมีรูปหูแหลม รูปหน้ากลม และมีหนวด

ที่โหนดแต่ละโหนด เราคำนวณข้อมูลที่ได้รับสำหรับแต่ละคุณลักษณะ จากนั้นแยกโหนดตามคุณลักษณะที่มีข้อมูลที่ได้รับสูงกว่า โดยการเปรียบเทียบเอนโทรปีของโหนดกับเอนโทรปีที่มีน้ำหนักในสองโหนดที่แยกออก

ดังนั้น โหนดรูทจึงมีสัตว์ทุกตัวในชุดข้อมูลของเรา โปรดจำไว้ว่า $p_1^{node}$ คือสัดส่วนของคลาสบวก (แมว) ในโหนดรูท ดังนั้น

$$p_1^{node} = \frac{5}{10} = 0.5$$

ตอนนี้มาเขียนฟังก์ชันเพื่อคำนวณเอนโทรปีกัน

In [ ]:
def entropy(p):
    if p == 0 or p == 1:
        return 0
    else:
        return -p * np.log2(p) - (1- p)*np.log2(1 - p)
    
print(entropy(0.5))

เพื่อเป็นการอธิบาย มาคำนวณข้อมูลที่ได้รับหากเราแยกโหนดสำหรับแต่ละคุณลักษณะ เพื่อทำเช่นนี้ มาเขียนฟังก์ชันบางอย่างกัน

In [ ]:
def split_indices(X, index_feature):
    """Given a dataset and a index feature, return two lists for the two split nodes, the left node has the animals that have 
    that feature = 1 and the right node those that have the feature = 0 
    index feature = 0 => ear shape
    index feature = 1 => face shape
    index feature = 2 => whiskers
    """
    left_indices = []
    right_indices = []
    for i,x in enumerate(X):
        if x[index_feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices

ดังนั้น หากเราเลือก Ear Shape เพื่อแยกข้อมูล เราจะต้องมีในโหนดซ้าย (ตรวจสอบตารางด้านบน) ดัชนี: 
$$0 \quad 3 \quad 4 \quad 5 \quad 7$$

และดัชนีทางขวา คือส่วนที่เหลือ

In [ ]:
split_indices(X_train, 0)

ตอนนี้เราต้องการฟังก์ชันอื่นเพื่อคำนวณเอนโทรปีแบบถ่วงน้ำหนักในโหนดที่แบ่งออก ตามที่คุณได้เห็นในวิดีโอการบรรยาย เราต้องหา:

- $w^{\text{left}}$ และ  $w^{\text{right}}$, ซึ่งเป็นสัดส่วนของสัตว์ใน แต่ละโหนด **each node**.
- $p^{\text{left}}$ และ  $p^{\text{right}}$, ซึ่งเป็นสัดส่วนของแมวใน แต่ละส่วนที่แบ่ง **each split**.

โปรดสังเกตความแตกต่างระหว่างคำจำกัดความทั้งสองนี้! เพื่อเป็นการอธิบาย หากเราแบ่งโหนดรูทตามคุณลักษณะของดัชนี 0 (รูปหู) แล้วในโหนดซ้าย ซึ่งมีสัตว์ 0, 3, 4, 5 และ 7 เราจะมี:

$$w^{\text{left}}= \frac{5}{10} = 0.5 \text{ and } p^{\text{left}} = \frac{4}{5}$$
$$w^{\text{right}}= \frac{5}{10} = 0.5 \text{ and } p^{\text{right}} = \frac{1}{5}$$

In [ ]:
def weighted_entropy(X,y,left_indices,right_indices):
    """
    This function takes the splitted dataset, the indices we chose to split and returns the weighted entropy.
    """
    w_left = len(left_indices)/len(X)
    w_right = len(right_indices)/len(X)
    p_left = sum(y[left_indices])/len(left_indices)
    p_right = sum(y[right_indices])/len(right_indices)
    
    weighted_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)
    return weighted_entropy

In [ ]:
left_indices, right_indices = split_indices(X_train, 0)
weighted_entropy(X_train, y_train, left_indices, right_indices)

ดังนั้น เอ็นโทรปีแบบถ่วงน้ำหนักในโหนดที่แยกออก 2 โหนดคือ 0.72 เพื่อคำนวณ **Information Gain** เราต้องลบออกจากเอ็นโทรปีในโหนดที่เราเลือกที่จะแยก (ในกรณีนี้ โหนดรูท)

In [ ]:
def information_gain(X, y, left_indices, right_indices):
    """
    Here, X has the elements in the node and y is theirs respectives classes
    """
    p_node = sum(y)/len(y)
    h_node = entropy(p_node)
    w_entropy = weighted_entropy(X,y,left_indices,right_indices)
    return h_node - w_entropy

In [ ]:
information_gain(X_train, y_train, left_indices, right_indices)

ตอนนี้ มาคำนวณข้อมูลที่ได้จากการแยกโหนดรูทสำหรับแต่ละคุณลักษณะกัน

In [ ]:
for i, feature_name in enumerate(['Ear Shape', 'Face Shape', 'Whiskers']):
    left_indices, right_indices = split_indices(X_train, i)
    i_gain = information_gain(X_train, y_train, left_indices, right_indices)
    print(f"Feature: {feature_name}, information gain if we split the root node using this feature: {i_gain:.2f}")
    

ดังนั้น คุณสมบัติที่ดีที่สุดในการแยกคือรูปร่างหู จัดการโค้ดด้านล่างเพื่อดูการแยกในแอคชัน คุณไม่จำเป็นต้องเข้าใจบล็อกโค้ดต่อไปนี้

In [ ]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=1, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

กระบวนการนี้เป็นแบบเรียกซ้ำ **recursive** ซึ่งหมายความว่าเราต้องดำเนินการคำนวณเหล่านี้สำหรับแต่ละโหนดจนกว่าเราจะพบเกณฑ์การหยุด:

- หากความลึกของต้นไม้หลังการแยกเกินเกณฑ์
- หากโหนดผลลัพธ์มีคลาสเพียง 1 คลาส
- หากการเพิ่มข้อมูลของการแยกต่ำกว่าเกณฑ์

ต้นไม้สุดท้ายมีลักษณะดังนี้:

In [ ]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=2, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

ขอแสดงความยินดี! คุณทำสมุดบันทึกเสร็จแล้ว!